# Advanced Problems with Solutions: Python Built-In Iterables and Iterators

This notebook expands the topic of Python's built-in iterables, iterators, lazy evaluation, and streaming data processing.

## Learning goals

By the end, you should be able to:

- Distinguish an **iterable** from an **iterator**
- Predict when an object can be reused and when it is exhausted
- Build lazy pipelines with `map`, `filter`, generator expressions, `zip`, `enumerate`, and `itertools`
- Stream data without loading everything into memory
- Write custom iterables and iterators correctly
- Debug subtle iterator bugs


## Setup

All examples are self-contained. No external CSV file is required.


In [1]:
from __future__ import annotations

from collections import Counter, deque
from collections.abc import Callable, Iterable, Iterator
from dataclasses import dataclass
from io import StringIO
from itertools import (
    chain,
    count,
    cycle,
    dropwhile,
    filterfalse,
    groupby,
    islice,
    pairwise,
    repeat,
    takewhile,
    tee,
    zip_longest,
)
from operator import itemgetter
from typing import Any, TypeVar

T = TypeVar("T")
U = TypeVar("U")


## Quick reference: best practices

| Situation | Prefer | Avoid |
|---|---|---|
| You need one pass over large data | iterator / generator | `list(...)` just to loop once |
| You need to reuse data many times | store a sequence or use an iterator factory | reusing an exhausted iterator |
| You need the first `n` values | `itertools.islice(iterator, n)` | manual index counters everywhere |
| You need to pair values | `zip`, `pairwise`, `enumerate` | fragile index arithmetic |
| You are reading a file | iterate over the file object | `readlines()` for huge files |
| You need custom iteration | make `__iter__` return a fresh iterator for reusable containers | returning `self` unless the object is intentionally single-use |


# Section 1 — Protocol examples


In [2]:
# Iterable but not iterator:
r = range(5)
print("range has __iter__:", hasattr(r, "__iter__"))
print("range has __next__:", hasattr(r, "__next__"))

# Iterator:
r_iter = iter(r)
print("range iterator has __iter__:", hasattr(r_iter, "__iter__"))
print("range iterator has __next__:", hasattr(r_iter, "__next__"))

# An iterator is usually self-iterating:
print(iter(r_iter) is r_iter)


range has __iter__: True
range has __next__: False
range iterator has __iter__: True
range iterator has __next__: True
True


In [3]:
# Built-in functions often create lazy iterator objects.
z = zip([1, 2, 3], "abc")
m = map(str.upper, ["a", "b", "c"])
f = filter(lambda x: x % 2 == 0, range(10))
e = enumerate("xyz")

for name, obj in [("zip", z), ("map", m), ("filter", f), ("enumerate", e)]:
    print(name, type(obj), "iterator?", iter(obj) is obj)


zip <class 'zip'> iterator? True
map <class 'map'> iterator? True
filter <class 'filter'> iterator? True
enumerate <class 'enumerate'> iterator? True


# Problem 1 — Build an iterator inspector

Write a function `inspect_iteration(obj)` that returns a dictionary with:

- `type_name`
- `is_iterable`
- `is_iterator`
- `iter_returns_self`
- `first_three`, a preview of up to three items

Important constraint: your function should not crash when the object is not iterable.

Best-practice note: previewing an iterator consumes values. That is acceptable here because this function is diagnostic.


In [4]:
def inspect_iteration(obj: Any) -> dict[str, Any]:
    try:
        iterator = iter(obj)
    except TypeError:
        return {
            "type_name": type(obj).__name__,
            "is_iterable": False,
            "is_iterator": False,
            "iter_returns_self": False,
            "first_three": None,
        }

    return {
        "type_name": type(obj).__name__,
        "is_iterable": True,
        "is_iterator": hasattr(obj, "__next__"),
        "iter_returns_self": iterator is obj,
        "first_three": list(islice(iterator, 3)),
    }


samples = [
    42,
    [10, 20, 30, 40],
    range(10),
    iter(range(10)),
    zip([1, 2, 3], "abc"),
    enumerate("python"),
]

for sample in samples:
    print(inspect_iteration(sample))


{'type_name': 'int', 'is_iterable': False, 'is_iterator': False, 'iter_returns_self': False, 'first_three': None}
{'type_name': 'list', 'is_iterable': True, 'is_iterator': False, 'iter_returns_self': False, 'first_three': [10, 20, 30]}
{'type_name': 'range', 'is_iterable': True, 'is_iterator': False, 'iter_returns_self': False, 'first_three': [0, 1, 2]}
{'type_name': 'range_iterator', 'is_iterable': True, 'is_iterator': True, 'iter_returns_self': True, 'first_three': [0, 1, 2]}
{'type_name': 'zip', 'is_iterable': True, 'is_iterator': True, 'iter_returns_self': True, 'first_three': [(1, 'a'), (2, 'b'), (3, 'c')]}
{'type_name': 'enumerate', 'is_iterable': True, 'is_iterator': True, 'iter_returns_self': True, 'first_three': [(0, 'p'), (1, 'y'), (2, 't')]}


# Problem 2 — Diagnose iterator exhaustion

A teammate writes this function and expects it to return the same numbers twice:

```python
def duplicate_pass(values):
    return list(values), list(values)
```

Explain the bug and write two safe alternatives:

1. `duplicate_pass_materialized(values)`: good when the data is small
2. `duplicate_pass_factory(make_values)`: good when the data is large or single-use


In [5]:
def duplicate_pass(values: Iterable[T]) -> tuple[list[T], list[T]]:
    return list(values), list(values)


print("With list:", duplicate_pass([1, 2, 3]))
print("With iterator:", duplicate_pass(iter([1, 2, 3])))


def duplicate_pass_materialized(values: Iterable[T]) -> tuple[list[T], list[T]]:
    data = list(values)
    return data.copy(), data.copy()


def duplicate_pass_factory(make_values: Callable[[], Iterable[T]]) -> tuple[list[T], list[T]]:
    return list(make_values()), list(make_values())


assert duplicate_pass_materialized(iter([1, 2, 3])) == ([1, 2, 3], [1, 2, 3])
assert duplicate_pass_factory(lambda: iter([1, 2, 3])) == ([1, 2, 3], [1, 2, 3])

print("materialized:", duplicate_pass_materialized(iter(range(5))))
print("factory:", duplicate_pass_factory(lambda: range(5)))


With list: ([1, 2, 3], [1, 2, 3])
With iterator: ([1, 2, 3], [])
materialized: ([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])
factory: ([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])


# Problem 3 — Prove lazy evaluation with side effects

Create a lazy pipeline that:

1. Starts from numbers `0..9`
2. Prints when a number is generated
3. Squares each number
4. Keeps only squares greater than `20`
5. Takes the first three matching results

The goal is to show that the whole input is **not** computed upfront.


In [6]:
def noisy_numbers(n: int) -> Iterator[int]:
    for x in range(n):
        print(f"generating {x}")
        yield x


pipeline = (
    x_squared
    for x in noisy_numbers(10)
    for x_squared in [x * x]
    if x_squared > 20
)

print("Pipeline created. Nothing has been generated yet.")
first_three = list(islice(pipeline, 3))
print("first_three:", first_three)


Pipeline created. Nothing has been generated yet.
generating 0
generating 1
generating 2
generating 3
generating 4
generating 5
generating 6
generating 7
first_three: [25, 36, 49]


# Problem 4 — Safe pairing with `zip`

You receive two columns: usernames and scores.

Tasks:

1. Show that normal `zip` silently truncates to the shortest input.
2. Write `strict_pairs(left, right)` that raises an error when lengths differ.
3. Write `padded_pairs(left, right, fillvalue=None)` that keeps all values.


In [7]:
users = ["ana", "bo", "cy", "di"]
scores = [91, 84]

print("normal zip silently truncates:", list(zip(users, scores)))


def strict_pairs(left: Iterable[T], right: Iterable[U]) -> list[tuple[T, U]]:
    return list(zip(left, right, strict=True))


def padded_pairs(
    left: Iterable[T],
    right: Iterable[U],
    fillvalue: Any = None,
) -> list[tuple[Any, Any]]:
    return list(zip_longest(left, right, fillvalue=fillvalue))


try:
    print(strict_pairs(users, scores))
except ValueError as exc:
    print("strict_pairs error:", exc)

print("padded:", padded_pairs(users, scores, fillvalue="MISSING"))


normal zip silently truncates: [('ana', 91), ('bo', 84)]
strict_pairs error: zip() argument 2 is shorter than argument 1
padded: [('ana', 91), ('bo', 84), ('cy', 'MISSING'), ('di', 'MISSING')]


# Problem 5 — Streaming CSV rows

You are given car data as text. The first row is the header and the second row contains type labels.

Tasks:

1. Build `car_rows(csv_text)` as a generator of dictionaries.
2. Skip the type row.
3. Convert numeric fields.
4. Compute average MPG by origin without storing all rows.


In [8]:
CARS_CSV = """Car;MPG;Cylinders;Displacement;Horsepower;Weight;Acceleration;Model;Origin
STRING;DOUBLE;INT;DOUBLE;DOUBLE;DOUBLE;DOUBLE;INT;CAT
Chevrolet Chevelle Malibu;18.0;8;307.0;130.0;3504.;12.0;70;US
Buick Skylark 320;15.0;8;350.0;165.0;3693.;11.5;70;US
Toyota Corolla Mark ii;24.0;4;113.0;95.0;2372.;15.0;70;Japan
Volkswagen 1131 Deluxe Sedan;26.0;4;97.0;46.0;1835.;20.5;70;Europe
Datsun PL510;27.0;4;97.0;88.0;2130.;14.5;70;Japan
Ford Pinto;25.0;4;98.0;0;2046.;19.0;71;US
Volkswagen Super Beetle 117;0;4;97.0;48.0;1978.;20.0;71;Europe
Honda Civic CVCC;33.0;4;91.0;53.0;1795.;17.5;75;Japan
Volkswagen Rabbit Custom Diesel;43.1;4;90.0;48.0;1985.;21.5;78;Europe
Mazda GLC Deluxe;32.8;4;78.0;52.0;1985.;19.4;78;Japan
"""


def convert_car_row(raw: dict[str, str]) -> dict[str, Any]:
    return {
        "Car": raw["Car"],
        "MPG": float(raw["MPG"]),
        "Cylinders": int(raw["Cylinders"]),
        "Displacement": float(raw["Displacement"]),
        "Horsepower": float(raw["Horsepower"]),
        "Weight": float(raw["Weight"]),
        "Acceleration": float(raw["Acceleration"]),
        "Model": int(raw["Model"]),
        "Origin": raw["Origin"],
    }


def car_rows(csv_text: str) -> Iterator[dict[str, Any]]:
    stream = StringIO(csv_text)
    header = next(stream).rstrip("\n").split(";")
    _type_row = next(stream)  # Skip type labels.

    for line in stream:
        values = line.rstrip("\n").split(";")
        raw = dict(zip(header, values, strict=True))
        yield convert_car_row(raw)


def average_mpg_by_origin(rows: Iterable[dict[str, Any]]) -> dict[str, float]:
    totals: dict[str, float] = {}
    counts: dict[str, int] = {}

    for row in rows:
        # Treat MPG=0 as missing data in this dataset.
        if row["MPG"] == 0:
            continue
        origin = row["Origin"]
        totals[origin] = totals.get(origin, 0.0) + row["MPG"]
        counts[origin] = counts.get(origin, 0) + 1

    return {origin: totals[origin] / counts[origin] for origin in totals}


print(next(car_rows(CARS_CSV)))
print(average_mpg_by_origin(car_rows(CARS_CSV)))


{'Car': 'Chevrolet Chevelle Malibu', 'MPG': 18.0, 'Cylinders': 8, 'Displacement': 307.0, 'Horsepower': 130.0, 'Weight': 3504.0, 'Acceleration': 12.0, 'Model': 70, 'Origin': 'US'}
{'US': 19.333333333333332, 'Japan': 29.2, 'Europe': 34.55}


# Problem 6 — Write reusable iterator utilities

Implement these lazy utilities:

- `take(n, iterable)`
- `drop(n, iterable)`
- `nth(iterable, index, default=None)`
- `sliding_window(iterable, size)`

Use `itertools` where it improves clarity.


In [9]:
def take(n: int, iterable: Iterable[T]) -> list[T]:
    if n < 0:
        raise ValueError("n must be non-negative")
    return list(islice(iterable, n))


def drop(n: int, iterable: Iterable[T]) -> Iterator[T]:
    if n < 0:
        raise ValueError("n must be non-negative")
    return islice(iterable, n, None)


def nth(iterable: Iterable[T], index: int, default: T | None = None) -> T | None:
    if index < 0:
        raise ValueError("index must be non-negative")
    return next(islice(iterable, index, None), default)


def sliding_window(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    it = iter(iterable)
    window = deque(islice(it, size), maxlen=size)

    if len(window) == size:
        yield tuple(window)

    for item in it:
        window.append(item)
        yield tuple(window)


assert take(3, count(10)) == [10, 11, 12]
assert list(drop(2, [10, 20, 30, 40])) == [30, 40]
assert nth("abcdef", 2) == "c"
assert nth("abc", 99, default="missing") == "missing"
assert list(sliding_window([1, 2, 3, 4, 5], 3)) == [(1, 2, 3), (2, 3, 4), (3, 4, 5)]

print("All utility tests passed.")


All utility tests passed.


# Problem 7 — Chunk an iterable lazily

Write `chunked(iterable, size)` that yields tuples of length `size`, except the final tuple may be shorter.

Example:

```python
list(chunked(range(10), 4))
# [(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]
```

Constraint: do not convert the whole iterable to a list.


In [10]:
def chunked(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)

    while True:
        chunk = tuple(islice(iterator, size))
        if not chunk:
            return
        yield chunk


assert list(chunked(range(10), 4)) == [(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]
assert list(chunked("abcdef", 2)) == [("a", "b"), ("c", "d"), ("e", "f")]
assert list(chunked([], 3)) == []

print(list(chunked(range(13), 5)))


[(0, 1, 2, 3, 4), (5, 6, 7, 8, 9), (10, 11, 12)]


# Problem 8 — Pairwise deltas

You have a stream of sensor readings. Compute the difference between each consecutive reading.

Tasks:

1. Use `itertools.pairwise`.
2. Write a compatible implementation using `tee` for environments where `pairwise` is unavailable.
3. Explain why `tee` should be used carefully with very large, unevenly consumed iterators.


In [11]:
readings = [10.0, 10.5, 9.75, 11.25, 11.0]

deltas_builtin = [b - a for a, b in pairwise(readings)]
print("pairwise deltas:", deltas_builtin)


def pairwise_compatible(iterable: Iterable[T]) -> Iterator[tuple[T, T]]:
    first, second = tee(iterable)
    next(second, None)
    return zip(first, second)


deltas_compatible = [b - a for a, b in pairwise_compatible(readings)]
assert deltas_compatible == deltas_builtin

print("compatible deltas:", deltas_compatible)

# Best-practice note:
# tee stores values internally when the duplicated iterators are consumed at different speeds.
# That hidden cache can grow large, so prefer pairwise() or a small explicit buffer when possible.


pairwise deltas: [0.5, -0.75, 1.5, -0.25]
compatible deltas: [0.5, -0.75, 1.5, -0.25]


# Problem 9 — Custom reusable iterable vs single-use iterator

Build two countdown classes:

1. `ReusableCountdown(n)` should be iterable many times.
2. `SingleUseCountdown(n)` should be its own iterator and become exhausted.

Then demonstrate the behavioral difference.


In [12]:
@dataclass(frozen=True)
class ReusableCountdown:
    start: int

    def __iter__(self) -> Iterator[int]:
        current = self.start
        while current >= 0:
            yield current
            current -= 1


class SingleUseCountdown:
    def __init__(self, start: int) -> None:
        self.current = start

    def __iter__(self) -> "SingleUseCountdown":
        return self

    def __next__(self) -> int:
        if self.current < 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


reusable = ReusableCountdown(3)
single_use = SingleUseCountdown(3)

print("Reusable pass 1:", list(reusable))
print("Reusable pass 2:", list(reusable))

print("Single-use pass 1:", list(single_use))
print("Single-use pass 2:", list(single_use))


Reusable pass 1: [3, 2, 1, 0]
Reusable pass 2: [3, 2, 1, 0]
Single-use pass 1: [3, 2, 1, 0]
Single-use pass 2: []


# Problem 10 — Sentinel iteration with `iter(callable, sentinel)`

Python's `iter` can turn a zero-argument callable into an iterator that stops when a sentinel value is returned.

Task: read fixed-size chunks from a stream until `""` is returned.


In [13]:
text_stream = StringIO("abcdefghijklmnopqrstuvwxyz")

chunks = list(iter(lambda: text_stream.read(5), ""))
print(chunks)

assert chunks == ["abcde", "fghij", "klmno", "pqrst", "uvwxy", "z"]


def read_chunks(text: str, chunk_size: int) -> Iterator[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    stream = StringIO(text)
    yield from iter(lambda: stream.read(chunk_size), "")


print(list(read_chunks("lazy iteration", 4)))


['abcde', 'fghij', 'klmno', 'pqrst', 'uvwxy', 'z']
['lazy', ' ite', 'rati', 'on']


# Problem 11 — `groupby` correctly

`itertools.groupby` groups only **consecutive** equal keys. It does not automatically group all matching values across the entire iterable.

Tasks:

1. Show the pitfall.
2. Sort before grouping when you need global grouping.
3. Compute car count by origin.


In [14]:
cars = list(car_rows(CARS_CSV))

# Pitfall: groupby only sees consecutive equal keys.
unsorted_origins = [row["Origin"] for row in cars]
print("origins in original order:", unsorted_origins)

consecutive_groups = {
    origin: len(list(group))
    for origin, group in groupby(cars, key=itemgetter("Origin"))
}
print("consecutive groups:", consecutive_groups)

# Correct global grouping: sort by the same key first.
cars_sorted = sorted(cars, key=itemgetter("Origin"))
global_groups = {
    origin: len(list(group))
    for origin, group in groupby(cars_sorted, key=itemgetter("Origin"))
}
print("global groups:", global_groups)

assert sum(global_groups.values()) == len(cars)


origins in original order: ['US', 'US', 'Japan', 'Europe', 'Japan', 'US', 'Europe', 'Japan', 'Europe', 'Japan']
consecutive groups: {'US': 1, 'Japan': 1, 'Europe': 1}
global groups: {'Europe': 3, 'Japan': 4, 'US': 3}


# Problem 12 — Build a lazy log-processing pipeline

You are given application log lines.

Tasks:

1. Parse each line into a dictionary.
2. Keep only `ERROR` lines.
3. Extract the service name.
4. Count errors by service.
5. Keep the pipeline lazy until the final `Counter`.


In [15]:
LOG_TEXT = """2026-06-22T08:00:01 INFO api request_id=1 ok
2026-06-22T08:00:02 ERROR worker request_id=2 timeout
2026-06-22T08:00:03 WARNING api request_id=3 slow
2026-06-22T08:00:04 ERROR api request_id=4 database
2026-06-22T08:00:05 INFO worker request_id=5 ok
2026-06-22T08:00:06 ERROR scheduler request_id=6 missed_job
2026-06-22T08:00:07 ERROR api request_id=7 timeout
"""


def lines(text: str) -> Iterator[str]:
    yield from StringIO(text)


def parse_log_line(line: str) -> dict[str, str]:
    timestamp, level, service, *message_parts = line.strip().split()
    return {
        "timestamp": timestamp,
        "level": level,
        "service": service,
        "message": " ".join(message_parts),
    }


parsed_rows = map(parse_log_line, lines(LOG_TEXT))
error_rows = filter(lambda row: row["level"] == "ERROR", parsed_rows)
error_services = map(itemgetter("service"), error_rows)
counts = Counter(error_services)

print(counts)
assert counts == Counter({"api": 2, "worker": 1, "scheduler": 1})


Counter({'api': 2, 'worker': 1, 'scheduler': 1})


# Problem 13 — Implement a `Peekable` iterator

Create a wrapper that lets you look at the next value without consuming it.

Required methods:

- `peek(default=None)`
- `__iter__`
- `__next__`

The wrapper should work with any iterable.


In [16]:
class Peekable(Iterator[T]):
    def __init__(self, iterable: Iterable[T]) -> None:
        self._iterator = iter(iterable)
        self._cache: deque[T] = deque()

    def __iter__(self) -> "Peekable[T]":
        return self

    def __next__(self) -> T:
        if self._cache:
            return self._cache.popleft()
        return next(self._iterator)

    def peek(self, default: T | None = None) -> T | None:
        if not self._cache:
            try:
                self._cache.append(next(self._iterator))
            except StopIteration:
                return default
        return self._cache[0]


p = Peekable(range(3))

assert p.peek() == 0
assert p.peek() == 0
assert next(p) == 0
assert next(p) == 1
assert p.peek() == 2
assert list(p) == [2]
assert p.peek("done") == "done"

print("Peekable works.")


Peekable works.


# Problem 14 — Advanced challenge: merge sorted streams lazily

You have multiple sorted iterables. Merge them into one sorted output stream.

Use `heapq.merge`, which is lazy and does not materialize all inputs.


In [17]:
from heapq import merge

stream_a = (x for x in [1, 4, 7, 10])
stream_b = (x for x in [2, 3, 9])
stream_c = (x for x in [0, 5, 6, 8])

merged = merge(stream_a, stream_b, stream_c)

print("first five:", list(islice(merged, 5)))
print("remaining:", list(merged))

# Notice that `merged` is an iterator. It continues from where islice stopped.


first five: [0, 1, 2, 3, 4]
remaining: [5, 6, 7, 8, 9, 10]


# Problem 15 — Debug a subtle consumption bug

The function below tries to report the first bad record and then process all records. But it accidentally loses data because `next(records)` consumes an item.

Fix it using `Peekable`.


In [18]:
def validate_then_process_wrong(records: Iterator[int]) -> list[int]:
    first = next(records, None)
    if first is not None and first < 0:
        raise ValueError("first record is negative")
    return [x * 10 for x in records]


print("wrong result:", validate_then_process_wrong(iter([1, 2, 3])))


def validate_then_process(records: Iterable[int]) -> list[int]:
    stream = Peekable(records)

    first = stream.peek()
    if first is not None and first < 0:
        raise ValueError("first record is negative")

    return [x * 10 for x in stream]


assert validate_then_process([1, 2, 3]) == [10, 20, 30]

try:
    validate_then_process([-1, 2, 3])
except ValueError as exc:
    print("correctly rejected:", exc)


wrong result: [20, 30]
correctly rejected: first record is negative


# Problem 16 — Build a mini iterator toolkit

Implement a small toolkit of lazy functions:

- `flatten(nested)`
- `compact(iterable)` removes falsey values
- `first_true(iterable, predicate, default=None)`
- `consume(iterable, n=None)` advances an iterator quickly

Then test each function.


In [19]:
def flatten(nested: Iterable[Iterable[T]]) -> Iterator[T]:
    yield from chain.from_iterable(nested)


def compact(iterable: Iterable[T]) -> Iterator[T]:
    return filter(None, iterable)


def first_true(
    iterable: Iterable[T],
    predicate: Callable[[T], bool],
    default: T | None = None,
) -> T | None:
    return next(filter(predicate, iterable), default)


def consume(iterable: Iterable[T], n: int | None = None) -> None:
    iterator = iter(iterable)

    if n is None:
        deque(iterator, maxlen=0)
    else:
        next(islice(iterator, n, n), None)


assert list(flatten([[1, 2], [3], [], [4, 5]])) == [1, 2, 3, 4, 5]
assert list(compact([0, 1, "", "hello", [], [1]])) == [1, "hello", [1]]
assert first_true(range(10), lambda x: x > 6) == 7
assert first_true(range(3), lambda x: x > 6, default=-1) == -1

it = iter(range(10))
consume(it, 4)
assert next(it) == 4

print("Toolkit tests passed.")


Toolkit tests passed.


# Problem 17 — Infinite iterators safely

Use infinite iterators without creating infinite loops.

Tasks:

1. Generate even numbers forever with `count`.
2. Take the first 10.
3. Use `takewhile` to keep values below 50.
4. Combine `cycle` and `islice` to create a repeating pattern.


In [20]:
evens = (2 * x for x in count(0))
first_10_evens = list(islice(evens, 10))
print(first_10_evens)

under_50 = list(takewhile(lambda x: x < 50, (3 * x for x in count(0))))
print(under_50)

pattern = list(islice(cycle(["red", "green", "blue"]), 8))
print(pattern)

assert first_10_evens == [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
assert pattern == ["red", "green", "blue", "red", "green", "blue", "red", "green"]


[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
[0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48]
['red', 'green', 'blue', 'red', 'green', 'blue', 'red', 'green']


# Problem 18 — Final case study: streaming top cars

Using the CSV stream from earlier:

Tasks:

1. Stream rows lazily.
2. Ignore missing MPG values.
3. Keep cars with at least 30 MPG.
4. Return the top `n` cars by MPG.
5. Keep the transformation pipeline readable.


In [21]:
def valid_mpg_rows(rows: Iterable[dict[str, Any]]) -> Iterator[dict[str, Any]]:
    return filter(lambda row: row["MPG"] > 0, rows)


def high_mpg_rows(rows: Iterable[dict[str, Any]], minimum_mpg: float = 30.0) -> Iterator[dict[str, Any]]:
    return filter(lambda row: row["MPG"] >= minimum_mpg, rows)


def top_cars_by_mpg(csv_text: str, n: int = 5) -> list[dict[str, Any]]:
    rows = car_rows(csv_text)
    rows = valid_mpg_rows(rows)
    rows = high_mpg_rows(rows, minimum_mpg=30.0)

    # Sorting materializes the filtered result, but only after lazy filtering.
    return sorted(rows, key=itemgetter("MPG"), reverse=True)[:n]


for car in top_cars_by_mpg(CARS_CSV, n=5):
    print(f'{car["Car"]:<35} {car["MPG"]:>5.1f} MPG  {car["Origin"]}')


Volkswagen Rabbit Custom Diesel      43.1 MPG  Europe
Honda Civic CVCC                     33.0 MPG  Japan
Mazda GLC Deluxe                     32.8 MPG  Japan


# Extra practice prompts

Try solving these without looking back:

1. Write `count_lines(text)` using `sum(1 for _ in StringIO(text))`.
2. Write `only_unique_consecutive(iterable)` using `groupby`.
3. Write `roundrobin(*iterables)` that alternates values from each iterable until all are exhausted.
4. Write `batched_strict(iterable, n)` that raises `ValueError` if the final batch is incomplete.
5. Write `windowed_average(iterable, size)` using `sliding_window`.


In [22]:
def count_lines(text: str) -> int:
    return sum(1 for _ in StringIO(text))


def only_unique_consecutive(iterable: Iterable[T]) -> Iterator[T]:
    for key, _group in groupby(iterable):
        yield key


def roundrobin(*iterables: Iterable[T]) -> Iterator[T]:
    iterators = deque(iter(it) for it in iterables)

    while iterators:
        iterator = iterators.popleft()
        try:
            yield next(iterator)
        except StopIteration:
            continue
        iterators.append(iterator)


def batched_strict(iterable: Iterable[T], n: int) -> Iterator[tuple[T, ...]]:
    for batch in chunked(iterable, n):
        if len(batch) != n:
            raise ValueError("incomplete final batch")
        yield batch


def windowed_average(iterable: Iterable[float], size: int) -> Iterator[float]:
    for window in sliding_window(iterable, size):
        yield sum(window) / size


assert count_lines("a\nb\nc\n") == 3
assert list(only_unique_consecutive("AAABCCDAA")) == ["A", "B", "C", "D", "A"]
assert list(roundrobin("ABC", [1, 2], "xyzw")) == ["A", 1, "x", "B", 2, "y", "C", "z", "w"]
assert list(batched_strict(range(6), 3)) == [(0, 1, 2), (3, 4, 5)]
assert list(windowed_average([10, 20, 30, 40], 2)) == [15.0, 25.0, 35.0]

try:
    list(batched_strict(range(5), 2))
except ValueError as exc:
    print("strict batching error:", exc)

print("Extra practice solutions passed.")


strict batching error: incomplete final batch
Extra practice solutions passed.


# Key takeaways

- An **iterable** can produce an iterator with `iter(obj)`.
- An **iterator** implements `__next__` and is usually consumed once.
- Many built-ins are lazy: `zip`, `map`, `filter`, `enumerate`, file objects, and generator expressions.
- Lazy pipelines are memory-efficient but can hide consumption bugs.
- Use `itertools` to express iterator logic clearly and safely.
- For custom container-like objects, `__iter__` should usually return a fresh iterator.
